## Loopcode for analysing GRF_data

In [ ]:
from scipy.io import loadmat
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Filter import butter_lowpass_filter
import os

print(os.getcwd())

/Users/ernstdavidts/Library/CloudStorage/OneDrive-UGent/Master I/Oslo/Curved_Running_Pilot/Base


In [ ]:
def get_GRK_data(mat_path, force_plate = "FP1"):
    
    data = loadmat(mat_path)
    qtmkey = mat_path.split("/")[-1]

    qtm = data[f"qtm_{qtmkey}"]

    FP1 = qtm.Force[0]
    FP2 = qtm.Force[1]

    if force_plate == "FP1":
        Fx = -(FP1.Force[0])
        Fy = -(FP1.Force[1])
        Fz = -(FP1.Force[2])
    else:
        Fx = -(FP2.Force[0])
        Fy = -(FP2.Force[1])
        Fz = -(FP2.Force[2])
    
    # correction + filter
    correction = np.median(Fz[Fz<50])
    Fx = Fx - correction
    Fy = Fy - correction
    Fz = Fz - correction

    Fx = butter_lowpass_filter(Fx, cutoff= 120, fs= 1000, order= 4)
    Fy = butter_lowpass_filter(Fy, cutoff= 120, fs= 1000, order= 4)
    Fz = butter_lowpass_filter(Fz, cutoff= 120, fs= 1000, order= 4)
    
    Fz = Fz[Fz < 20]
    return [Fx, Fy, Fz]


In [ ]:
def GRF_segm_ct(data):
    Fx = data[0]
    Fy = data[1]
    Fz = data[2]
    x = np.arange(len(data[0]))*0.001

    # Make control lines to let it work all the time like when there is no clear IC (when in 25_Curve_T1)
    IC = [i for i in range(1, len(Fz)) if Fz[i] != 0 and (Fz[i-1] == 0)][0]
    TO = [i for i in range(0, len(Fz)) if Fz[i] == 0 and (Fz[i-1] != 0)][0]

    segment_Fz = Fz[IC:TO]
    segment_Fy = Fy[IC:TO]
    segment_Fx = Fx[IC:TO]
    contacttijd = TO - IC

    return [segment_Fx, segment_Fy, segment_Fz] , contacttijd

In [ ]:
# adapt so that it's able to do this for multiple files in the loop! 

def GRF_stats(segment, force="Fz"):

    VILR = np.max(np.diff(segment))

    if force == "Fz":
        return {
        "mean": np.mean(segment),
        "max": np.max(segment),
        "VILR": VILR
        }
    elif force == "Fx":
        return {
        "mean": np.mean(segment),
        "max": np.max(segment),
        }
    else:
        return {
        "PBF": np.min(segment),
        }